# Goal

Серия экспериментов. Треним за 5 поколений по 6 млн шагов с последующим отбором чемпионов. Наследники этих чемпионов будут использоваться на следующем шаге.

Данный эксперимент - это второй шаг. Даём агенту играть с девятью жизнями с начала.

Тут используем претрейнед `VisionHead` и `RenderHead` из `WorldModel`, `is_trainable=True`. 

# set_hyperparameters

In [1]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    generations_count = 5
    generation_ind = 1
    generation_steps_count = 6_000_000
    all_generations_steps_count = generation_steps_count * generations_count
    learn_rate_range = (0.00025, 0.00025 * 0.1)
    ent_coef_range = (0.05, 0.05 * 0.1)
    ####
    
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    HP.agent.parent = dict(agent=optuna_trial.suggest_categorical('agent.parent', [
        '18n_ppo_tr_frostbite_02:9',
        '18n_ppo_tr_frostbite_02:11',
        '18n_ppo_tr_frostbite_02:17',
    ])) 
    HP.agent.sequence_length = 4 # length observation chain agent incepts
    HP.agent.ob_shape = (1, 178, 152) 
    HP.agent.action_plan_length = 10 # number of actions agent must think upfront about
    HP.agent.vision_head = dict(grid=(6, 6), features_counts=(16, 32, 64, 128), is_trainable=True)
    HP.agent.action_plan_lv_encoding = 'separate'
    HP.agent.is_causal_action_plan = True
    HP.agent.d_model = 256 # dimension of the transformer
    HP.agent.layers_count = 3 # number of transformer layers
    HP.agent.heads_count = 4 # number of heads used in multi-head attention
    HP.agent.attention_backend = 'EFFICIENT_ATTENTION'
    HP.agent.prediction_target = 'next_obs'
    HP.agent.render_heads = dict(heads_count=4, is_trainable=True)
    
    # Video params
    HP.video.capture_policy = 'every(500000)' # video capture policy depending on steps
    HP.video.capture_env_rams = None
    HP.video.capture_env_ram_patches = None
    HP.video.break_on_level_passed = False
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = generation_steps_count # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = None
    HP.ppo.rollout_env_ram_patches = [
        ['nine_lives'],
    ]
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 256 
    learn_rate_change_speed = (learn_rate_range[1] - learn_rate_range[0]) / generations_count
    learn_rate_a = learn_rate_range[0] + learn_rate_change_speed * generation_ind
    learn_rate_b = learn_rate_a + learn_rate_change_speed
    HP.ppo.learn_rate = f'linear({learn_rate_a}, {learn_rate_b})'
    # HP.ppo.learn_rate = 'linear(0.00025, 0.00015)'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.vf_coef = 0.5 # coefficient of the value function within loss function
    ent_coef_change_speed = (ent_coef_range[1] - ent_coef_range[0]) / generations_count
    ent_coef_a = ent_coef_range[0] + ent_coef_change_speed * generation_ind
    ent_coef_b = ent_coef_a + ent_coef_change_speed
    HP.ppo.ent_coef = f'linear({ent_coef_a}, {ent_coef_b})' # coefficient of the entropy member within loss function
    # HP.ppo.ent_coef = 'linear(0.05, 0.03)' # coefficient of the entropy member within loss function
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.1
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP
# @launchit.stop

# Results
<TBD>

`is_trainable=True` не помогло. По-прежнему агент едва проходит два уровня. И это при этом, что на первом шаге он обнадёживал, т.к. шёл близко к 17-ой серии.

<img src="./img/levels_passed.png">
<img src="./img/reward.png">
<img src="./img/episode_r.png">

Это очень странно. Заинтересовало, что гораздо больше, чем в 17-ой серии "прострелов" - это когда агент на видео мы видим, что агент проходит по 15 уровней. Хотя на самом деле игра переходит в режим сохранения экрана (https://share.google/aimode/27PlUFMZg8WEpCBWR). Происходит это, когда последняя жизнь заканчивается не быстрой гибелью, а из-за температуры (агент замерзает на последней жизни). В этом случае игра не останавливается, а продолжается. И самое противное, что там подключается встроенный робот. В итоге агент вроде видит экраны, нажимает кнопки, но ничего не происходит. И это просто в хлам разрушает политику. Агент познаёт безнадёгу.

Фикс простой:
```c++
(mine) misha@thinkbook:~/dev/mine/ale$ git diff src/ale/games/supported/Frostbite.cpp
diff --git a/src/ale/games/supported/Frostbite.cpp b/src/ale/games/supported/Frostbite.cpp
index 188e01a..c7a97c1 100644
--- a/src/ale/games/supported/Frostbite.cpp
+++ b/src/ale/games/supported/Frostbite.cpp
@@ -52,7 +52,8 @@ void FrostbiteSettings::step(const System& system) {
   // higher values & properly decrement, but we do not gain lives beyond 9.
   int lives_byte = (readRam(&system, 0xCC) & 0xF);
   int flag = readRam(&system, 0xF1) & 0x80;
-  m_terminal = (lives_byte == 0 && flag != 0);
+  int temperature = readRam(&system, 101);
+  m_terminal = (lives_byte == 0 && flag != 0) || (lives_byte == 0 && temperature == 0);
 
   m_lives = lives_byte + 1;
 }
```

Т.е. если при роллауте какой-то бедолага попадаёт в это состояние, то он может там застрять если не навечно, то на очень долго. И он будет отравлять всем (другим окружениям) жизнь. И это поведение безнадёги может как болезнь распространится. И это видно по графикам, когда вдруг ни с того, ни с сего value_mean/episode_r вдруг падают камнем вниз.

**Выводы**
1) проблема режима "сохранения экрана" - явная наведённая проблема. Степень влияния может быть огромной.
2) надо перетренироваывать агента, начиная с 1-ого шага на новом ale_py, чтобы убрать наведённую проблему.
3) потом уже разбираться, где болит